In [6]:
"""
Load pcos_model.pkl + pcos_features.pkl and print the model's accuracy on the rows it never trained on.

Put these three files in the same folder as this script:
    pcos_model.pkl
    pcos_features.pkl
    PCOS_data.csv        (the 541-row Kaggle file)

Run:
    python check_models.py
    python check_models.py path\\to\\PCOS_data.csv
"""

import sys
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")  # old scikit-learn version warnings are harmless here

CSV = sys.argv[1] if len(sys.argv) > 1 else "PCOS_data.csv"

# ---- 1. load the two pickles ----
model = joblib.load("pcos_model.pkl")
feats = joblib.load("pcos_features.pkl")
print("Loaded model:", type(model).__name__, "| trees:", len(model.estimators_), "| features:", len(feats))

# ---- 2. load + clean the data exactly like the training notebook ----
df = pd.read_csv(CSV)
df.columns = df.columns.str.strip()
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")] + ["Sl. No", "Patient File No."])
df["AMH(ng/mL)"] = pd.to_numeric(df["AMH(ng/mL)"].replace("a", np.nan))
df["II    beta-HCG(mIU/mL)"] = pd.to_numeric(df["II    beta-HCG(mIU/mL)"].replace("1.99.", "1.99"))

y = df["PCOS (Y/N)"].astype(int)
X = df.drop(columns=["PCOS (Y/N)"])
X = X.fillna({
    "Marraige Status (Yrs)": X["Marraige Status (Yrs)"].median(),
    "AMH(ng/mL)": X["AMH(ng/mL)"].median(),
    "Fast food (Y/N)": X["Fast food (Y/N)"].mode()[0],
})

# ---- 3. the same 80/20 split the model was trained with (seed 42) ----
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Rows: {len(X)} total | {len(Xtr)} the model trained on | {len(Xte)} it never saw\n")

# ---- 4. accuracies ----
for name, Xs, ys in [("TRAIN (rows it learned from - not a real score)", Xtr, ytr),
                     ("TEST  (rows it never saw - the honest score)", Xte, yte)]:
    p = model.predict_proba(Xs[feats])[:, 1]
    pred = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(ys, pred).ravel()
    print(name)
    print(f"  Accuracy     : {accuracy_score(ys, pred):.4f}  ({tp + tn} of {len(ys)} correct)")
    print(f"  Sensitivity  : {tp / (tp + fn):.4f}  (PCOS caught: {tp} of {tp + fn})")
    print(f"  Specificity  : {tn / (tn + fp):.4f}  (normal cleared: {tn} of {tn + fp})")
    print(f"  AUC          : {roc_auc_score(ys, p):.4f}\n")

Loaded model: RandomForestClassifier | trees: 700 | features: 41


FileNotFoundError: [Errno 2] No such file or directory: '-f'

In [10]:
"""
Load pcos_model.pkl + pcos_features.pkl and print the model's accuracy on the rows it never trained on.

Put these three files in the same folder as this script:
    pcos_model.pkl
    pcos_features.pkl
    PCOS_data.csv        (the 541-row Kaggle file)

Run (Anaconda Prompt):   python check_models.py
or paste the whole file into one Jupyter cell and run it. It finds the 541-row CSV by itself.
"""

import glob
import os
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")  # old scikit-learn version warnings are harmless here



def find_csv():
    """The 541-row PCOS file, found in the current folder (ignores Jupyter's own '-f ...' arguments)."""
    candidates = [a for a in sys.argv[1:] if a.lower().endswith(".csv") and os.path.isfile(a)]
    candidates += ["PCOS_data.csv"] + sorted(glob.glob("*.csv"))
    seen, report = set(), []
    for f in candidates:
        if f in seen or not os.path.isfile(f):
            continue
        seen.add(f)
        try:
            d = pd.read_csv(f)
        except Exception:
            continue
        d.columns = d.columns.str.strip()
        report.append(f"{f} ({len(d)} rows)")
        if len(d) == 541 and "PCOS (Y/N)" in d.columns:
            return f
    raise SystemExit(
        f"Could not find the 541-row PCOS_data.csv in {os.getcwd()}\n"
        f"CSV files seen here: {report or 'none'}\n"
        "Copy PCOS_data.csv into this folder and run again."
    )


CSV = find_csv()
print("Using data file:", CSV, "| folder:", os.getcwd())

# ---- 1. load the two pickles ----
model = joblib.load("pcos_model.pkl")
feats = joblib.load("pcos_features.pkl")
print("Loaded model:", type(model).__name__, "| trees:", len(model.estimators_), "| features:", len(feats))

# ---- 2. load + clean the data exactly like the training notebook ----
df = pd.read_csv(CSV)
df.columns = df.columns.str.strip()
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")] + ["Sl. No", "Patient File No."])
df["AMH(ng/mL)"] = pd.to_numeric(df["AMH(ng/mL)"].replace("a", np.nan))
df["II    beta-HCG(mIU/mL)"] = pd.to_numeric(df["II    beta-HCG(mIU/mL)"].replace("1.99.", "1.99"))

y = df["PCOS (Y/N)"].astype(int)
X = df.drop(columns=["PCOS (Y/N)"])
X = X.fillna({
    "Marraige Status (Yrs)": X["Marraige Status (Yrs)"].median(),
    "AMH(ng/mL)": X["AMH(ng/mL)"].median(),
    "Fast food (Y/N)": X["Fast food (Y/N)"].mode()[0],
})

# ---- 3. the same 80/20 split the model was trained with (seed 42) ----
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Rows: {len(X)} total | {len(Xtr)} the model trained on | {len(Xte)} it never saw\n")

# ---- 4. accuracies ----
for name, Xs, ys in [("TRAIN (rows it learned from - not a real score)", Xtr, ytr),
                     ("TEST  (rows it never saw - the honest score)", Xte, yte)]:
    p = model.predict_proba(Xs[feats])[:, 1]
    pred = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(ys, pred).ravel()
    print(name)
    print(f"  Accuracy     : {accuracy_score(ys, pred):.4f}  ({tp + tn} of {len(ys)} correct)")
    print(f"  Sensitivity  : {tp / (tp + fn):.4f}  (PCOS caught: {tp} of {tp + fn})")
    print(f"  Specificity  : {tn / (tn + fp):.4f}  (normal cleared: {tn} of {tn + fp})")
    print(f"  AUC          : {roc_auc_score(ys, p):.4f}\n")

Using data file: PCOS_data.csv | folder: C:\Users\chakr\Desktop\PCOS\PCOS-final\backend\model_weights\clinical
Loaded model: RandomForestClassifier | trees: 700 | features: 41
Rows: 541 total | 432 the model trained on | 109 it never saw

TRAIN (rows it learned from - not a real score)
  Accuracy     : 0.9977  (431 of 432 correct)
  Sensitivity  : 0.9929  (PCOS caught: 140 of 141)
  Specificity  : 1.0000  (normal cleared: 291 of 291)
  AUC          : 1.0000

TEST  (rows it never saw - the honest score)
  Accuracy     : 0.9083  (99 of 109 correct)
  Sensitivity  : 0.7500  (PCOS caught: 27 of 36)
  Specificity  : 0.9863  (normal cleared: 72 of 73)
  AUC          : 0.9486

